# UniChart — the `bar()` method, end to end

`nb.bar()` is the unified interface for bar charts. One method, three views, plus
overlays, styling, scaling and layout controls. This notebook walks through every
way to use it:

| Section | What it shows |
|---|---|
| 1 | Basic call — one y variable |
| 2 | Multiple y variables (subplot grid) |
| 3 | `barmode` — `'group'` vs `'stack'` |
| 4 | `by='sets'` — one subplot per dataset |
| 5 | `by='dataset_x'` — datasets as the x-axis, with `agg` |
| 6 | `markers=` overlays and positional pairing |
| 7 | `color=` — forcing a single bar color |
| 8 | Titles, axis labels, footer |
| 9 | `scale()` — axis limits, including overlay columns |
| 10 | Layout — `figsize`, `ncols`/`nrows`, `suppress_legends` |
| 11 | Persistent defaults and state memory |
| 12 | `darkmode` |

For a deep dive on overlay *styles* (`marker` / `tick` / `whisker`, stem dashes),
see `bar_whisker_demo.ipynb` in this folder.

In [1]:
# --- make repo-root importable (notebook lives in demo_notebooks/) ---
import sys, os
_repo_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

import numpy as np
import pandas as pd
from unichart import UnichartNotebook

PHASES = ['TAKEOFF', 'CLIMB', 'CRUISE', 'DESCENT']

def engine_df(seed, base_egt):
    """One row per flight phase: temperatures, ratios and their limit columns."""
    r = np.random.default_rng(seed)
    return pd.DataFrame({
        'PHASE':     PHASES,
        'EGT':       base_egt + r.uniform(-40, 40, 4).round(1),
        'RU':        (base_egt * 0.12 + r.uniform(-8, 8, 4)).round(1),
        'N1':        (92 + r.uniform(-6, 4, 4)).round(1),
        'EGT_LIMIT': [950, 920, 880, 860],
        'RU_LIMIT':  [115, 118, 120, 112],
    })

def build():
    """Fresh notebook with three engines loaded."""
    nb = UnichartNotebook()
    nb.load_df(engine_df(1, 830), title='Engine A (ESN 771201)')
    nb.load_df(engine_df(2, 860), title='Engine B (ESN 771305)')
    nb.load_df(engine_df(3, 845), title='Engine C (ESN 771442)')
    return nb

---
## 1. Basic call — one y variable

The default view is `by='vars'`: one subplot per y variable, one bar color per
dataset, grouped side-by-side within each x category.

In [2]:
nb = build()
nb.bar(x='PHASE', y='EGT')

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A (ESN 771201)
Loaded Set 1: Engine B (ESN 771305)
Loaded Set 2: Engine C (ESN 771442)


## 2. Multiple y variables

Pass a list to `y=` and each variable gets its own subplot (its own y scale).
The x-axis title only appears on each column's bottom-most panel, and the
subplot titles name the variables — no repeated axis labels.

In [3]:
nb = build()
nb.bar(x='PHASE', y=['EGT', 'RU', 'N1'])

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A (ESN 771201)
Loaded Set 1: Engine B (ESN 771305)
Loaded Set 2: Engine C (ESN 771442)


## 3. `barmode` — `'group'` (default) vs `'stack'`

`barmode='stack'` stacks the datasets' bars instead of placing them
side-by-side. (It falls back to the `set_default_format` default when not
passed — see section 11.)

In [4]:
nb = build()
nb.bar(x='PHASE', y='EGT', barmode='stack', suptitle='Stacked EGT by phase')

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A (ESN 771201)
Loaded Set 1: Engine B (ESN 771305)
Loaded Set 2: Engine C (ESN 771442)


## 4. `by='sets'` — one subplot per dataset

The per-dataset view flips the encoding: each dataset gets its own panel and
**color encodes the variable** instead of the dataset. `var_format` overrides
the color/alpha of individual variables here (`by='datasets'` is an alias).

In [5]:
nb = build()
nb.var_format('EGT', color='firebrick')
nb.bar(x='PHASE', y=['EGT', 'RU', 'N1'], by='sets')

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A (ESN 771201)
Loaded Set 1: Engine B (ESN 771305)
Loaded Set 2: Engine C (ESN 771442)


## 5. `by='dataset_x'` — datasets as the x-axis

One single chart where the x categories are the *datasets* and the bars are the
y variables, each scaled to its own y-axis (colored to match its bars). Since a
dataset may hold many rows, `agg` picks the reduction: `'mean'` (default),
`'median'`, `'sum'`, `'max'`, `'min'`, `'first'`, `'last'`. Hovering a bar
reports the aggregate as `<agg>: <value> (n=<points>)`, where *n* counts the
valid (non-NaN) rows that fed it.

In [6]:
nb = build()
nb.bar(y=['EGT', 'RU'], by='dataset_x', agg='max',
       suptitle='Per-engine maxima (each variable on its own axis)')

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A (ESN 771201)
Loaded Set 1: Engine B (ESN 771305)
Loaded Set 2: Engine C (ESN 771442)


## 6. `markers=` — overlay columns on the bars

An overlay column draws a glyph at its value on top of each dataset's own bar.
Unstyled overlays use the dataset color and a cycling symbol; style one via
`var_format` (color, marker, size, and `style='tick'` / `'whisker'` for
limit-line rendering — see `bar_whisker_demo.ipynb`).

In [7]:
nb = build()
nb.var_format('EGT_LIMIT', color='red', style='tick')
nb.bar(x='PHASE', y='EGT', markers='EGT_LIMIT')

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A (ESN 771201)
Loaded Set 1: Engine B (ESN 771305)
Loaded Set 2: Engine C (ESN 771442)


### Positional pairing with multiple y variables

With `y` and `markers` both lists, overlays pair positionally: the i-th marker
column draws on the i-th y variable's panel, attached to that variable's bars
(extras fall back to the first panel). Each limit stays on its own variable's
scale.

In [8]:
nb = build()
nb.var_format('EGT_LIMIT', color='red', style='whisker', linestyle='--')
nb.var_format('RU_LIMIT', color='purple', style='tick')
nb.bar(x='PHASE', y=['EGT', 'RU'], markers=['EGT_LIMIT', 'RU_LIMIT'])

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A (ESN 771201)
Loaded Set 1: Engine B (ESN 771305)
Loaded Set 2: Engine C (ESN 771442)


## 7. `color=` — force a single bar color

`color` overrides every dataset's color in the default `by='vars'` view (useful
when the datasets are distinguished by the subplot instead). The `by='sets'`
and `by='dataset_x'` views color per variable and ignore it.

In [9]:
nb = build()
nb.bar(x='PHASE', y='EGT', color='steelblue', suptitle='Single-color bars')

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A (ESN 771201)
Loaded Set 1: Engine B (ESN 771305)
Loaded Set 2: Engine C (ESN 771442)


## 8. Titles, axis labels, footer

- `suptitle=` sets the figure title per call; `nb.suptitle = ...` persists it.
- `nb.x_label` / `nb.y_label` override the axis titles (x on outer panels only,
  y on the first column).
- `footer=` pins a caption below the figure; `nb.footer = ...` persists it.

In [10]:
nb = build()
nb.x_label = 'Flight phase'
nb.y_label = 'EGT (\u00b0C)'
nb.bar(x='PHASE', y='EGT',
       suptitle='Fleet EGT by phase',
       footer='Source: synthetic demo data \u2014 one row per phase per engine.')

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A (ESN 771201)
Loaded Set 1: Engine B (ESN 771305)
Loaded Set 2: Engine C (ESN 771442)


## 9. `scale()` — axis limits

`nb.scale(column, (lo, hi))` pins that variable's axis in every later plot.
Overlay columns are accepted too: since an overlay shares its paired bar's
y-axis, its range is *unioned* with the bar variable's own scale so both the
bars and the limit glyphs stay in frame.

In [11]:
nb = build()
nb.var_format('EGT_LIMIT', color='red', style='tick')
nb.scale('EGT', (700, 900))        # bars alone would clip the 950 limit...
nb.scale('EGT_LIMIT', (700, 960))  # ...the union keeps the limit visible
nb.bar(x='PHASE', y='EGT', markers='EGT_LIMIT')

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A (ESN 771201)
Loaded Set 1: Engine B (ESN 771305)
Loaded Set 2: Engine C (ESN 771442)
Limits set for 'EGT': (700, 900)
Limits set for 'EGT_LIMIT': (700, 960)


## 10. Layout — `figsize`, `ncols`/`nrows`, `suppress_legends`

- `figsize=(w, h)` in inches (100 px per inch); `nb.figsize = ...` persists it.
- `ncols`/`nrows` force the subplot grid (default: near-square, max 3 columns).
- `suppress_legends=True` starts every trace hidden (`legendonly`) so the
  reader toggles on just what they want from the legend.

In [12]:
nb = build()
nb.bar(x='PHASE', y=['EGT', 'RU', 'N1'], ncols=1, figsize=(9, 12),
       suptitle='One column, taller figure')

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A (ESN 771201)
Loaded Set 1: Engine B (ESN 771305)
Loaded Set 2: Engine C (ESN 771442)


In [13]:
nb = build()
nb.bar(x='PHASE', y='EGT', suppress_legends=True,
       suptitle='Click legend entries to reveal each engine')

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A (ESN 771201)
Loaded Set 1: Engine B (ESN 771305)
Loaded Set 2: Engine C (ESN 771442)


## 11. Persistent defaults and state memory

`set_default_format` stores fallbacks for `barmode`, `agg` and
`suppress_legends` so you don't repeat them per call. And every plot call
remembers `x`/`y` (`nb.last_x` / `nb.last_y`), so a follow-up `nb.bar()` with
no arguments re-plots the same columns — handy after `select`/`omit`.

In [14]:
nb = build()
nb.set_default_format(barmode='stack')
nb.bar(x='PHASE', y='EGT', suptitle='barmode picked up from the stored default')

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A (ESN 771201)
Loaded Set 1: Engine B (ESN 771305)
Loaded Set 2: Engine C (ESN 771442)


In [15]:
nb.omit(1)      # hide Engine B...
nb.bar()        # ...and re-plot the same x/y without re-specifying them

## 12. `darkmode`

`nb.darkmode = True` switches every later figure to the dark template.

In [16]:
nb = build()
nb.darkmode = True
nb.bar(x='PHASE', y='EGT', suptitle='Dark template')

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A (ESN 771201)
Loaded Set 1: Engine B (ESN 771305)
Loaded Set 2: Engine C (ESN 771442)


---
## Where to go next

- **Overlay styles in depth** (`tick`, `whisker`, stem dashes, validation):
  `bar_whisker_demo.ipynb`
- **Fonts, plot size, static PNG export**: `demo_plot_size_and_layout.ipynb`,
  `static_images_demo.ipynb`
- The same `by` vocabulary (`'vars'`, `'sets'`, `'dataset_x'`) drives
  `nb.box()` and `nb.histogram()` too.